# Notebook 3: The SGET Architecture

This notebook walks through how SGET is structured so you can read, modify, and extend the codebase. We'll cover the central model, the signal-based communication pattern, the layer configuration system, the 2D layout, and the GUI widget tree.

## Prerequisites
- Notebooks 1 and 2 completed
- Neo4j running with the example DSG loaded (Notebook 2 does this)

## 1. Architecture Overview

SGET follows a **model-signal-view** pattern (Qt's version of MVC):

```
 JSON file ──► spark_dsg ──► heracles bulk load ──► Neo4j
                                                      │
                                              SceneGraphModel
                                           (cache + Qt signals)
                                        /      |       |        \
                                  GraphView  LayerPanel  PropertyPanel  SnapshotPanel
                                  (2D view)  (toggles)   (editor)      (save/restore)
```

**Key principle**: all data flows through the `SceneGraphModel`. Views never talk to Neo4j directly — they read from the model's in-memory cache and react to Qt signals when data changes.

**Why a cache?** Neo4j queries are fast, but not fast enough for rendering hundreds of nodes at 60fps. The model maintains a dict of node properties and a list of edges, populated from Neo4j on load. Mutations update both the cache and Neo4j, then emit signals.

**Why not keep spark_dsg in memory?** spark_dsg's pybind11 bindings are designed for bulk construction and serialization, not fine-grained mutation with change notification. The DSG object is transient — used only at the load/save boundaries.

## 2. The SceneGraphModel

The `SceneGraphModel` (a `QObject`) is the heart of the application. Let's see it in action — we'll create one, connect it to Neo4j, load a graph, and observe how signals fire.

In [ ]:
# Qt requires a QApplication instance for signals to work, even without a GUI.
import os

from heracles.utils import get_labelspace
from PySide6.QtWidgets import QApplication

from sget.backend.scene_graph_model import SceneGraphModel

app = QApplication.instance() or QApplication([])

model = SceneGraphModel()
model.connect("neo4j://127.0.0.1:7687", "neo4j", "neo4j_pw")

# Set labelspaces (same as app.py does from CLI args).
object_labels = get_labelspace("ade20k_mit_label_space.yaml")
room_labels = get_labelspace("b45_label_space.yaml")
model.set_labelspaces(
    object_labels={v: k for k, v in object_labels.items()},
    room_labels={v: k for k, v in room_labels.items()},
)

print(f"Connected: {model.connected}")

In [ ]:
# SignalSpy — a simple helper that records every emission of a Qt signal.
# This is the same pattern used in our test suite (tests/test_scene_graph_model.py).
class SignalSpy:
    def __init__(self, signal):
        self.calls = []
        signal.connect(lambda *args: self.calls.append(args))


# Watch the graph_loaded signal.
loaded_spy = SignalSpy(model.graph_loaded)

# Load the example DSG.
DSG_PATH = os.path.expanduser(
    "~/software/mit/sget/heracles/heracles/examples/scene_graphs/example_dsg.json"
)
model.load_from_json(DSG_PATH)

print(f"graph_loaded fired: {len(loaded_spy.calls)} time(s)")
print(f"Cache: {len(model.get_all_nodes())} nodes, {len(model.get_edges())} edges")

In [ ]:
# Now watch what happens when we add a node — the model updates the cache,
# writes to Neo4j, and emits node_added.
from heracles import constants

add_spy = SignalSpy(model.node_added)

model.add_node(
    constants.OBJECTS,
    "o(99)",
    {
        "pos_x": 0.0,
        "pos_y": 0.0,
        "pos_z": 0.0,
        "bbox_x": 0.0,
        "bbox_y": 0.0,
        "bbox_z": 0.0,
        "bbox_l": 1.0,
        "bbox_w": 1.0,
        "bbox_h": 1.0,
        "class": "box",
        "name": "demo_node",
    },
)

print(f"node_added fired: {add_spy.calls}")
print(f"Node in cache: {model.get_node('o(99)') is not None}")
print(f"Node layer:    {model.get_node_layer('o(99)')}")

# Clean up.
model.remove_node("o(99)")

## 3. Layer Configuration — Single Source of Truth

SGET uses `utils/colors.py` as the **single source of truth** for layer configuration. It defines `LAYER_STYLES` — an ordered list of `LayerStyle` dataclasses that specify each layer's label, ID, display name, color, and category character.

Everything else derives from this list: the model's `LAYER_ORDER`, the layer panel's rows, the graph view's node colors, and the spatial layout.

**Important**: the position in `LAYER_STYLES` defines the hierarchy order, NOT the raw layer IDs. MeshPlaces has `layer_id=20` but sits below Rooms (`layer_id=4`) in the hierarchy. Code that needs to compare hierarchy levels (e.g., detecting parent-child relationships) uses the index in `LAYER_STYLES`.

All widget files use `heracles.constants` (e.g., `hc.ROOMS`, `hc.OBJECTS`) for layer label strings instead of hardcoded `"Room"`, `"Object"` etc.

In [ ]:
from sget.utils.colors import LAYER_STYLES, STYLE_BY_LABEL

# LAYER_STYLES defines the hierarchy order (top to bottom in the UI).
for style in LAYER_STYLES:
    print(
        f"  {style.display_name:12s}  layer_id={style.layer_id:2d}  "
        f"label={style.layer_label:10s}  color={style.color}  char={style.category_char}"
    )

# Quick lookup by heracles label string.
print(f"\nRoom color: {STYLE_BY_LABEL['Room'].color}")

## 4. The 2D Layout

The graph view uses a **spatial layout** — nodes are positioned based on their actual 3D coordinates from the scene graph, projected onto the x-y plane. This means nodes that are near each other in the real environment appear near each other in the view.

The projection is: `scene_x = world_x * scale`, `scene_y = -world_y * scale` (y is negated so "north" is up, matching map conventions). The scale factor converts from meters to pixels.

The layout operates on the model's cache (plain dicts), not on spark_dsg objects.

In [ ]:
from sget.utils.layout import DEFAULT_SCALE, compute_layout

# The layout function takes the same data structures the model cache provides.
nodes = model.get_all_nodes()
node_layers = {ns: model.get_node_layer(ns) for ns in nodes}
edges = model.get_edges()

positions = compute_layout(nodes, node_layers, edges)

print(f"Computed positions for {len(positions)} nodes")
print(f"Scale factor: {DEFAULT_SCALE} pixels per world unit\n")

# Show a sample position and verify the projection.
sample_ns = list(positions.keys())[0]
sample_props = nodes[sample_ns]
scene_x, scene_y = positions[sample_ns]
wc = sample_props["center"]
print(f"Node {sample_ns}:")
print(f"  World pos:  [{wc[0]:.2f}, {wc[1]:.2f}, {wc[2]:.2f}]")
print(f"  Scene pos:  ({scene_x:.1f}, {scene_y:.1f})")
expected_x = wc[0] * DEFAULT_SCALE
expected_y = -wc[1] * DEFAULT_SCALE
print(f"  Expected:   ({expected_x:.1f}, {expected_y:.1f})")

## 5. The GUI Widget Tree

The GUI is built with **PySide6** (Qt for Python). Here's the full widget hierarchy:

```
MainWindow (QMainWindow)
├── Central Widget: GraphView (QWidget)
│   ├── QLineEdit (search bar — filters nodes by symbol/class/name)
│   └── _ZoomableGraphicsView (QGraphicsView)
│       └── QGraphicsScene
│           ├── NodeItem (QGraphicsEllipseItem) × N  [selectable, optionally draggable]
│           ├── EdgeItem (QGraphicsLineItem) × M     [selectable]
│           └── QGraphicsPolygonItem (boundary overlays for Rooms)
├── Left Dock: LayerPanel (QWidget)
│   ├── [Add Node] [Delete] buttons
│   ├── Per-layer rows: [checkbox] [swatch] [label] [count]
│   └── "Show CONTAINS edges" checkbox (off by default)
├── Right Dock: PropertyPanel (QWidget)
│   ├── "Locked" checkbox (per-node drag lock)
│   └── QFormLayout with editable fields + Apply button
├── Right Dock: SnapshotPanel (QWidget)
│   ├── "Save Snapshot" button
│   └── List of snapshots: [name] [timestamp] [counts] [Restore] [Delete]
└── Menu Bar
    ├── File: Open, Save As, Export Image, Connect, Refresh from DB, Quit
    ├── Edit: Add Node (Ctrl+N), Delete (Del), Group (Ctrl+G), Draw Region (Ctrl+R)
    └── View: Focus on Subtree (Ctrl+F), Show All (Esc), toggle dock panels
```

### Navigation
- **Click and drag** to pan (default mode)
- **Shift+drag** for rubber-band selection
- **Ctrl+click** for additive selection
- **Mouse wheel** to zoom
- **F** to fit the graph to the viewport

### Key interaction modes

**Normal mode** (default): click to select, pan by dragging, search bar filters nodes, `F` to fit view.

**Polygon draw mode** (Edit → Draw Region): click to place vertices, double-click to close, Escape to cancel. Captures nodes inside the polygon and opens the Group dialog.

**Focus mode** (View → Focus on Subtree): shows only the selected node and its descendants. View → Show All (Escape) to restore.

### Per-node drag-to-reposition
Each node has a "Locked" checkbox in the property panel (locked by default). Unchecking it allows that specific node to be dragged — edges follow in real time, and the new position is committed to Neo4j on mouse release.

### Signal flow for node selection
1. User clicks a `NodeItem` in the `QGraphicsScene`
2. `QGraphicsScene.selectionChanged` fires
3. `GraphView._on_scene_selection_changed()` reads the selected items
4. Calls `model.set_selection([node_symbols...])`
5. Model emits `selection_changed` signal
6. `PropertyPanel._on_selection_changed()` populates the form
7. `GraphView._on_selection_changed()` updates gold highlights (with a guard to prevent recursion)
8. `MainWindow._on_selection_changed()` shows count summary in status bar

### Incremental view updates
When a node or edge is added/removed/updated through the model, the graph view updates individual items:
- `model.node_added` → `GraphView._on_node_added()` creates a single `NodeItem`
- `model.node_removed` → `GraphView._on_node_removed()` removes the `NodeItem` and its connected `EdgeItem`s
- `model.node_updated` → `GraphView._on_node_updated()` moves the node and repositions edges
- `model.edge_added` / `edge_removed` → add/remove a single `EdgeItem`

Full layout recomputation only happens on `graph_loaded` (File → Open or Refresh from DB).

### Context menu (right-click in graph view)
- **1 parent + N children selected** → "Add N node(s) as children of R1"
- **2 different nodes selected** → "Add Edge" (self-edges prevented)
- **Nodes selected** → "Delete N Node(s)" (with confirmation)
- **Edges selected** → "Delete N Edge(s)" (with confirmation)

When rubber-band selects both nodes and edges, nodes take priority (edges are deselected).

## 6. Extending SGET

Here are practical pointers for common modifications:

**Adding a new property to the property panel**:
Edit `views/property_panel.py` → `_show_node()`. Add a widget for your property (QLineEdit, QSpinBox, etc.), store it in `self._widgets`, and handle it in `_on_apply()`. The key mapping `"pos" → "center"` is where widget names translate to Neo4j property names. Also add the new property name to `ALLOWED_PROPERTIES` in `neo4j_crud.py` (whitelist for Cypher injection prevention).

**Adding a new menu action**:
Edit `main_window.py` → `_setup_menus()`. Add a `QAction` to the appropriate menu and connect it to a handler method. See `_add_node`, `_group_selected`, `_draw_region`, `_focus_on_subtree`, and `_export_image` for examples.

**Adding a new dialog**:
Follow the pattern in `widgets/add_node_dialog.py` or `widgets/group_dialog.py`:
1. Subclass `QDialog` with form fields
2. Accept/reject buttons via `QDialogButtonBox`
3. A `get_result()` or `execute_*()` method that returns data on accept, None on cancel
4. Call from `main_window.py` handler

**Adding a new layer**:
1. Add a `LayerStyle` entry to `LAYER_STYLES` in `utils/colors.py`
2. Add a CREATE template function in `neo4j_crud.py` and register it in `_CREATE_DISPATCH`
3. Add the layer to `_PARENT_LAYERS` in `widgets/group_dialog.py` if it participates in grouping
4. Everything else (model, layout, layer panel, add node dialog) picks it up automatically
5. Use `heracles.constants` for the label string — don't hardcode

**Adding a new view or panel**:
Follow the pattern in `widgets/snapshot_panel.py`:
1. Create a widget class that takes the `SceneGraphModel` in its constructor
2. Connect to the model's signals (`graph_loaded`, `selection_changed`, `node_added`, etc.)
3. Add it as a dock widget in `main_window.py`
4. Add to the View menu via `view_menu.addAction(dock.toggleViewAction())`

**Adding a new interaction mode** (like the polygon tool):
1. Add state tracking in `GraphView` (e.g., `_polygon_mode_active`)
2. Override mouse events in `_ZoomableGraphicsView` to route to the mode
3. Emit a signal when the mode completes (e.g., `polygon_completed`)
4. Connect the signal in `main_window.py` to handle the result

**Subtree queries**:
`model.get_descendants(node_symbol)` does BFS over CONTAINS edges and returns all transitive children. Used by the Focus on Subtree feature.

**Snapshots**:
The `SnapshotPanel` stores JSON files in `.sget_snapshots/` next to the loaded file. An "initial_load" snapshot is auto-saved on first load. Each snapshot is a full scene graph export. Restoring a snapshot is equivalent to loading a new file.

**Chat agent integration**:
The chat agent (heracles_agents) connects to the same Neo4j database. After the agent modifies the graph, press Ctrl+Shift+R in SGET to refresh. Custom agent tools can be added by registering them in heracles_agents' tool registry.

**Security note**:
`neo4j_crud.py` validates property names against `ALLOWED_PROPERTIES` before building dynamic Cypher SET clauses. When adding new properties, add them to this whitelist.

**Running the test suite**:
```bash
pytest                                  # All 47 tests (requires Neo4j)
pytest tests/test_neo4j_crud.py         # CRUD layer (23 tests)
pytest tests/test_scene_graph_model.py  # Model tests (24 tests)
pytest -k "selection"                   # Tests matching a keyword
```

In [ ]:
# Clean up.
model.disconnect()
print("Done! You're ready to start contributing to SGET.")

## Summary

You've learned:
- SGET uses a **model-signal-view** pattern: `SceneGraphModel` is the single source of truth
- The model maintains a **dual store** (Neo4j + in-memory cache) and emits **Qt signals** on every mutation
- `utils/colors.py` is the **single source of truth** for layer configuration — everything derives from `LAYER_STYLES`. Hierarchy order is by position in the list, not raw layer IDs
- The **spatial layout** projects nodes onto the x-y plane using their real 3D coordinates
- **Navigation**: click-drag to pan, Shift+drag for rubber-band, F to fit, mouse wheel to zoom
- The GUI has dockable panels (Layers, Properties, Snapshots), a search bar, and three modes (normal, polygon draw, focus)
- **Per-node drag**: unlock a node's position in the property panel to drag it; edges follow in real time
- **Focus on subtree**: select a Room → Ctrl+F → shows only its descendants; Escape to restore
- **Context menu**: "Add as children", "Add Edge", "Delete" — prioritizes nodes over edges in mixed selections
- **Incremental updates**: node/edge add/remove/update signals update individual QGraphicsItems without full layout recompute
- **Snapshots**: save/restore named JSON snapshots; initial state auto-saved on first load
- **Chat agent**: heracles_agents TUI connects to the same Neo4j; Ctrl+Shift+R to refresh
- **Security**: property names validated against `ALLOWED_PROPERTIES` whitelist; use `heracles.constants` for layer strings
- To extend: add to `LAYER_STYLES` for layers, follow dialog/panel patterns for new UI, use `get_descendants()` for subtree queries

For the full implementation details, see the source files directly — they are thoroughly documented with design-choice comments.